In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    for block in message.content:
        if block.type == "text":
            return block.text

## 課題: コードグレーダー(構文検証)を作り、モデルグレーダーと統合する

- 対応: L14「コードベースの採点」。ページのプリフィル(```` ```code ````)は使わない(現行モデルでは使えない可能性が高い。L08)。
- 今回、新しく覚えること: **コードで出力を機械的に検証する**(JSON・Python・正規表現の構文チェック)。
- 目標: 各テストケースの出力に、構文のスコア(0 か 10)を付け、ex09 のモデルグレーダーの点と平均して、最終スコアにする。
- 前提: データセットの各ケースに、期待する形式(`format`)が必要。今回は、`dataset.json` を読んだあと、コードで足す(TODO 1)。
- API 呼び出し: 生成3回 + 採点3回 = 6回。少額の料金がかかる。TODO 2・3 は、API なしで確認できる。
- ヒントは、下の「ヒント」のセルにある。まず、見ずに、試してみる。

In [3]:
import ast
import json
import re
import statistics

# ex07 で作った dataset.json を読み込む(ex08・ex09 と同じ)
with open("../ex07_Generating-test-datasets/dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)
print(len(dataset))


# 構造化出力の応答から、本文を取り出す関数(ex06〜ex09 と同じ)
def get_text_from_response(response):
    for block in response.content:
        if block.type == "text":
            return block.text
    return None


# ex09 の採点結果のスキーマ(そのまま)
grade_schema = {
    "type": "object",
    "properties": {
        "strengths": {"type": "array", "items": {"type": "string"}},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
        "reasoning": {"type": "string"},
        "score": {"type": "integer"},
    },
    "required": ["strengths", "weaknesses", "reasoning", "score"],
    "additionalProperties": False,
}


# ex09 のモデルグレーダー(そのまま)
def grade_by_model(test_case, output):
    task = test_case["task"]
    eval_prompt = f"""
あなたは熟練のコードレビュアーです。AI が生成した次の解答を評価してください。

タスク: {task}
解答: {output}

次の項目を含めて評価してください。
- strengths: 良い点を1〜3個
- weaknesses: 改善点を1〜3個
- reasoning: 評価の理由を簡潔に
- score: 1〜10 の整数(10が最高、1が最低)
"""
    messages = []
    add_user_message(messages, eval_prompt)

    response = client.messages.create(
        model=model,
        max_tokens=4000,
        messages=messages,
        output_config={"format": {"type": "json_schema", "schema": grade_schema}},
    )

    text = get_text_from_response(response)
    return json.loads(text)

3


In [19]:
# ex08・ex09 の run_prompt(ベースライン版)。TODO 5 で、この文面を書き換える。
def run_prompt(test_case):
    """プロンプトとテストケースを結合して、Claude の出力を返す"""
    prompt = f"""
次のタスクを解決してください:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [5]:
# TODO 1: 各テストケースに、期待する形式("format")を足す
#   - 値は、"python"、"json"、"regex" のどれか。
#   - dataset の各 task を読んで、どの形式を求めているかを自分で判断して入れる。
#   - 足したあと、print で確認する。
for test_case in dataset:
    print(test_case["task"][:60])

# 例: dataset[0]["format"] = "python"
dataset[0]["format"] = "python"  # TODO
dataset[1]["format"] = "json"  # TODO
dataset[2]["format"] = "regex"  # TODO

print([tc["format"] for tc in dataset])

S3バケット名が命名規則（小文字・数字・ハイフンのみ、3〜63文字、先頭と末尾は英数字）を満たしているかを判定する単一の
AWS Lambda関数のIAMロールに付与する最小権限のポリシーを表す単一のJSONオブジェクトを作成してください。内
CloudWatch Logsのログ行から、AWS EC2インスタンスID（i-で始まり、8桁または17桁の16進数が続
['python', 'json', 'regex']


In [8]:
# TODO 2: 3つの構文検証関数を完成させる
#   - 解析に成功したら 10、失敗したら 0 を返す。
#   - JSON: json.loads(...)  失敗時の例外は json.JSONDecodeError
#   - Python: ast.parse(...)  失敗時の例外は SyntaxError
#   - 正規表現: re.compile(...)  失敗時の例外は re.error
#   - text は、前後の空白を取り除いてから解析する(.strip())。
def validate_json(text):
    try:
        json.loads(text.strip())  # TODO: text.strip() を JSON として解析する
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())  # TODO: text.strip() を Python として解析する
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())  # TODO: text.strip() を正規表現としてコンパイルする
        return 10
    except re.error:
        return 0

In [9]:
# TODO 2 の動作確認(API は呼ばない)。すべて通れば、何も表示されない。
assert validate_json('{"a": 1}') == 10
assert validate_json('```json\n{"a": 1}\n```') == 0
assert validate_python("def add(a, b):\n    return a + b") == 10
assert validate_python("```python\ndef add(a, b):\n    return a + b\n```") == 0
assert validate_regex("[a-z]+") == 10
assert validate_regex("[a-z") == 0
print("OK")

OK


In [10]:
# TODO 3: grade_syntax を完成させる
#   - test_case["format"] に応じて、使う検証関数を切り替える。
#   - "json" → validate_json、"python" → validate_python、"regex" → validate_regex
#   - 知らない形式のときは、エラーにして止める(L11 で学んだ、静かな失敗を避ける考え方)。
def grade_syntax(output, test_case):
    fmt = test_case["format"]
    if fmt == "json":
        return validate_json(output)  # TODO
    elif fmt == "python":
        return validate_python(output)  # TODO
    elif fmt == "regex":
        return validate_regex(output)  # TODO
    else:
        raise ValueError(f"未知の format: {fmt}")

In [11]:
# TODO 4: run_test_case を、2つの採点を統合する形に書き換える
#   - model_grade = grade_by_model(...)、model_score = その score
#   - syntax_score = grade_syntax(...)
#   - 最終スコア score = (model_score + syntax_score) / 2
#   - 戻り値に、"model_score" と "syntax_score" も入れる(あとで内訳を見るため)。
def run_test_case(test_case):
    """生成 → モデルグレーダー → コードグレーダー → 統合"""
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]  # TODO: model_grade から "score" を取り出す
    syntax_score = grade_syntax(output, test_case)  # TODO: grade_syntax を呼ぶ
    score = (model_score + syntax_score) / 2  # TODO: 2つの平均
    return {
        "output": output,
        "test_case": test_case,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": model_grade["reasoning"],
    }

In [20]:
# 実行(書き換えなくてよい)。内訳を見る。
def run_eval(dataset):
    results = []
    for test_case in dataset:
        results.append(run_test_case(test_case))
    average_score = statistics.mean(r["score"] for r in results)
    print(f"Average score: {average_score}")
    return results

results = run_eval(dataset)
for r in results:
    print("format:", r["test_case"]["format"],
          "| model:", r["model_score"],
          "| syntax:", r["syntax_score"],
          "| final:", r["score"])

Average score: 4.166666666666667
format: python | model: 8 | syntax: 0 | final: 4.0
format: json | model: 8 | syntax: 0 | final: 4.0
format: regex | model: 9 | syntax: 0 | final: 4.5


In [17]:
# TODO 5: プロンプトを具体的にして、もう一度評価する(この下に、新しい run_prompt を書く)
#   - 上の「ベースライン版」の run_prompt を、書き換えるか、同じ名前で再定義する。
#   - 追加する指示(日本語でよい):
#       「Python、JSON、または正規表現のみを返してください」
#       「コメント、解説、説明は付けないでください」
#   - プリフィルは使わない。
#   - 再定義したら、上の「実行」のセルをもう一度実行して、平均スコアと内訳を比べる。

def run_prompt(test_case):
    """プロンプトとテストケースを結合して、Claude の出力を返す"""
    prompt = f"""
次のタスクを解決してください:

{test_case["task"]}

Python、JSON、または正規表現のみを返してください
コメント、解説、説明、コードブロックの(```)を付けないでください

"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

## ヒント(困ったときだけ開く)

**TODO 1(format)**
- 各 task の文を読む。「Python 関数を書いてください」なら "python"、「JSON オブジェクトを作成してください」なら "json"、「正規表現を書いてください」なら "regex"。
- 順番は、`print` した 3 行で確かめる。

**TODO 2(検証関数)**
- JSON: `json.loads(text.strip())`
- Python: `ast.parse(text.strip())`
- 正規表現: `re.compile(text.strip())`
- `return 10` は、その下の行。例外が出たときだけ、`except` に入る。
- 動作確認のセルが、すべて通れば OK。通らないときは、どの `assert` で止まったかを見る。

**TODO 3(切り替え)**
- 返す値は、検証関数の戻り値。例: `return validate_json(output)`
- 引数の `output` は、検証する文字列。

**TODO 4(統合)**
- `model_score = model_grade["score"]`
- `syntax_score = grade_syntax(output, test_case)`
- 平均は、`(model_score + syntax_score) / 2`

**TODO 5(プロンプト)**
- 追加する指示を、f 文字列の中の `{test_case["task"]}` の前か後に足す。
- 出力にコードフェンス(```` ``` ````)が付くと、構文のスコアが 0 になる。その理由は、L14 の確認問題で見た通り。

**エラーが出たら**
- 最後の行を読む。`TypeError: unsupported operand` や `NoneType` は、どこかの TODO が `None` のまま、という意味のことが多い。
- `KeyError: 'format'` は、TODO 1 が未完了、という意味。

## 振り返り(自分の言葉で書く。短くてよい)

1. ベースラインで、`syntax_score` が 0 になったのは、どのケースか。その出力は、どんな形だったか。
2. 構文の点と、モデルの点を、等しい重みで平均した結果、どんなことが起きたか。重みを変えるなら、どんな場合か。
3. 正規表現の検証 (`re.compile`) は、どの程度、信頼できると思うか。普通の文章を渡すと、どうなるかを、試してみてよい。
4. TODO 5 で、プロンプトを具体的にした結果、平均スコアと内訳は、どう変わったか。